In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()

while ROOT != ROOT.parent:
    if (ROOT / "data_labeling").exists():
        break
    ROOT = ROOT.parent

VALIDATION_CSV = (
    ROOT / "data_labeling/data_labels/noaa/CL_Labels/validation_2024_2025.csv"
)

DETAILED_LABELS_CSV = (
    ROOT / "data_labeling/data_labels/noaa/labels_noaa_2010_2026.csv"
)

NOAA_EVENTS_CSV = (
    ROOT / "data_labeling/Catalogue/noaa_events.csv"
)

print("Repository:", ROOT)

Repository: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-


In [2]:
validation = pd.read_csv(VALIDATION_CSV)

In [3]:
validation

,label,goes_class
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1
...,...,...
8475,2025/06/30/HMI.m2025.06.30_19.00.00.jpg,0
8476,2025/06/30/HMI.m2025.06.30_20.00.00.jpg,0
8477,2025/06/30/HMI.m2025.06.30_21.00.00.jpg,0
8478,2025/06/30/HMI.m2025.06.30_22.00.00.jpg,0


In [4]:
validation["goes_class"] = validation["goes_class"].astype(int)

nf_count = (validation["goes_class"] == 0).sum()
fl_count = (validation["goes_class"] == 1).sum()

In [5]:
print("Total validation images:", len(validation))
print("No-flare images:", nf_count)
print("Flare images:", fl_count)
print("No-flare percentage:", round(100 * nf_count / len(validation), 2), "%")
print("Flare percentage:", round(100 * fl_count / len(validation), 2), "%")

Total validation images: 8480
No-flare images: 3321
Flare images: 5159
No-flare percentage: 39.16 %
Flare percentage: 60.84 %


In [6]:
detailed = pd.read_csv(DETAILED_LABELS_CSV)

In [7]:
validation_detailed = validation.merge(
    detailed[["label", "goes_class"]],
    on="label",
    how="left",
    suffixes=("_binary", "_exact"),
    validate="one_to_one"
)

print("Matched images:", validation_detailed["goes_class_exact"].notna().sum())
print("Missing exact classes:", validation_detailed["goes_class_exact"].isna().sum())

validation_detailed.head()

Matched images: 8480
Missing exact classes: 0


,label,goes_class_binary,goes_class_exact
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1,M2.1
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1,M2.1
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1,M2.1
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1,M2.1
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1,M2.1


In [8]:
validation_detailed

,label,goes_class_binary,goes_class_exact
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1,M2.1
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1,M2.1
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1,M2.1
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1,M2.1
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1,M2.1
...,...,...,...
8475,2025/06/30/HMI.m2025.06.30_19.00.00.jpg,0,C1.1
8476,2025/06/30/HMI.m2025.06.30_20.00.00.jpg,0,C1.1
8477,2025/06/30/HMI.m2025.06.30_21.00.00.jpg,0,C1.1
8478,2025/06/30/HMI.m2025.06.30_22.00.00.jpg,0,C1.1


In [9]:
flare_images = validation_detailed[
    validation_detailed["goes_class_binary"] == 1
].copy()

In [10]:
len(flare_images)

5159

In [11]:
flare_images

,label,goes_class_binary,goes_class_exact
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1,M2.1
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1,M2.1
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1,M2.1
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1,M2.1
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1,M2.1
...,...,...,...
8229,2025/06/20/HMI.m2025.06.20_13.00.00.jpg,1,M1.0
8230,2025/06/20/HMI.m2025.06.20_14.00.00.jpg,1,M1.0
8231,2025/06/20/HMI.m2025.06.20_15.00.00.jpg,1,M1.0
8232,2025/06/20/HMI.m2025.06.20_16.00.00.jpg,1,M1.0


In [12]:
flare_images["goes_class_exact"].value_counts()

goes_class_exact
M1.1    311
M1.0    274
M1.2    259
M1.6    253
M1.5    219
       ... 
M5.8      5
M5.2      1
M9.5      1
X1.0      1
M4.6      1
Name: count, Length: 84, dtype: int64

In [13]:
flare_images["category"] = (
    flare_images["goes_class_exact"]
    .astype(str)
    .str[0]
)

In [14]:
flare_images

,label,goes_class_binary,goes_class_exact,category
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1,M2.1,M
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1,M2.1,M
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1,M2.1,M
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1,M2.1,M
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1,M2.1,M
...,...,...,...,...
8229,2025/06/20/HMI.m2025.06.20_13.00.00.jpg,1,M1.0,M
8230,2025/06/20/HMI.m2025.06.20_14.00.00.jpg,1,M1.0,M
8231,2025/06/20/HMI.m2025.06.20_15.00.00.jpg,1,M1.0,M
8232,2025/06/20/HMI.m2025.06.20_16.00.00.jpg,1,M1.0,M


In [15]:
flare_images["category"].value_counts()

category
M    4528
X     631
Name: count, dtype: int64

In [16]:
m_count = (flare_images["category"] == "M").sum()
x_count = (flare_images["category"] == "X").sum()

In [17]:
print("Total flare images:", len(flare_images))
print("M-class images:", m_count)
print("X-class images:", x_count)

Total flare images: 5159
M-class images: 4528
X-class images: 631


In [18]:
m_images = flare_images[
    flare_images["category"] == "M"
].copy()

In [19]:
m_images

,label,goes_class_binary,goes_class_exact,category
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1,M2.1,M
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1,M2.1,M
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1,M2.1,M
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1,M2.1,M
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1,M2.1,M
...,...,...,...,...
8229,2025/06/20/HMI.m2025.06.20_13.00.00.jpg,1,M1.0,M
8230,2025/06/20/HMI.m2025.06.20_14.00.00.jpg,1,M1.0,M
8231,2025/06/20/HMI.m2025.06.20_15.00.00.jpg,1,M1.0,M
8232,2025/06/20/HMI.m2025.06.20_16.00.00.jpg,1,M1.0,M


In [20]:
m_images["magnitude"] = pd.to_numeric(
    m_images["goes_class_exact"].str.extract(
        r"^M(\d+(?:\.\d+)?)"
    )[0],
    errors="coerce"
)

In [21]:
m_images

,label,goes_class_binary,goes_class_exact,category,magnitude
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1,M2.1,M,2.1
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1,M2.1,M,2.1
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1,M2.1,M,2.1
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1,M2.1,M,2.1
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1,M2.1,M,2.1
...,...,...,...,...,...
8229,2025/06/20/HMI.m2025.06.20_13.00.00.jpg,1,M1.0,M,1.0
8230,2025/06/20/HMI.m2025.06.20_14.00.00.jpg,1,M1.0,M,1.0
8231,2025/06/20/HMI.m2025.06.20_15.00.00.jpg,1,M1.0,M,1.0
8232,2025/06/20/HMI.m2025.06.20_16.00.00.jpg,1,M1.0,M,1.0


In [22]:
m_images["subcategory"] = (
    "M"
    + np.floor(m_images["magnitude"])
    .astype(int)
    .astype(str)
)

m_images

,label,goes_class_binary,goes_class_exact,category,magnitude,subcategory
0,2024/07/01/HMI.m2024.07.01_00.00.00.jpg,1,M2.1,M,2.1,M2
1,2024/07/01/HMI.m2024.07.01_01.00.00.jpg,1,M2.1,M,2.1,M2
2,2024/07/01/HMI.m2024.07.01_02.00.00.jpg,1,M2.1,M,2.1,M2
3,2024/07/01/HMI.m2024.07.01_03.00.00.jpg,1,M2.1,M,2.1,M2
4,2024/07/01/HMI.m2024.07.01_04.00.00.jpg,1,M2.1,M,2.1,M2
...,...,...,...,...,...,...
8229,2025/06/20/HMI.m2025.06.20_13.00.00.jpg,1,M1.0,M,1.0,M1
8230,2025/06/20/HMI.m2025.06.20_14.00.00.jpg,1,M1.0,M,1.0,M1
8231,2025/06/20/HMI.m2025.06.20_15.00.00.jpg,1,M1.0,M,1.0,M1
8232,2025/06/20/HMI.m2025.06.20_16.00.00.jpg,1,M1.0,M,1.0,M1


In [23]:
m_subcategory_counts = (
    m_images["subcategory"]
    .value_counts()
    .rename_axis("M_subcategory")
    .reset_index(name="number_of_images")
)

m_subcategory_counts

,M_subcategory,number_of_images
0,M1,2043
1,M2,798
2,M3,571
3,M4,279
4,M5,254
5,M7,203
6,M6,143
7,M8,142
8,M9,95


In [24]:
m_subcategory_counts["order"] = (
    m_subcategory_counts["M_subcategory"]
    .str.replace("M", "", regex=False)
    .astype(int)
)

m_subcategory_counts

,M_subcategory,number_of_images,order
0,M1,2043,1
1,M2,798,2
2,M3,571,3
3,M4,279,4
4,M5,254,5
5,M7,203,7
6,M6,143,6
7,M8,142,8
8,M9,95,9


In [25]:
m_subcategory_counts = (
    m_subcategory_counts
    .sort_values("order")
    .drop(columns="order")
    .reset_index(drop=True)
)

m_subcategory_counts

,M_subcategory,number_of_images
0,M1,2043
1,M2,798
2,M3,571
3,M4,279
4,M5,254
5,M6,143
6,M7,203
7,M8,142
8,M9,95


Count X1, X2, X3, …

In [26]:
m_below_threshold = validation_detailed[
    validation_detailed["goes_class_exact"]
    .astype(str)
    .str.match(r"^M0\.")
].copy()

print("Sub-threshold M0.x image labels:", len(m_below_threshold))

display(
    m_below_threshold[
        ["label", "goes_class_binary", "goes_class_exact"]
    ]
)

# All M0.x samples must now be no-flare
assert (m_below_threshold["goes_class_binary"] == 0).all()

print("Passed: every M0.x sample has binary label 0.")

Sub-threshold M0.x image labels: 22


,label,goes_class_binary,goes_class_exact
4399,2025/01/10/HMI.m2025.01.10_01.00.00.jpg,0,M0.9
4400,2025/01/10/HMI.m2025.01.10_02.00.00.jpg,0,M0.9
4401,2025/01/10/HMI.m2025.01.10_03.00.00.jpg,0,M0.9
4402,2025/01/10/HMI.m2025.01.10_04.00.00.jpg,0,M0.9
4403,2025/01/10/HMI.m2025.01.10_05.00.00.jpg,0,M0.9
4404,2025/01/10/HMI.m2025.01.10_06.00.00.jpg,0,M0.9
4405,2025/01/10/HMI.m2025.01.10_07.00.00.jpg,0,M0.9
4406,2025/01/10/HMI.m2025.01.10_08.00.00.jpg,0,M0.9
4407,2025/01/10/HMI.m2025.01.10_09.00.00.jpg,0,M0.9
4408,2025/01/10/HMI.m2025.01.10_10.00.00.jpg,0,M0.9


Passed: every M0.x sample has binary label 0.


In [27]:
x_images = flare_images[
    flare_images["category"] == "X"
].copy()

x_images

,label,goes_class_binary,goes_class_exact,category
291,2024/07/13/HMI.m2024.07.13_03.00.00.jpg,1,X1.2,X
292,2024/07/13/HMI.m2024.07.13_04.00.00.jpg,1,X1.2,X
293,2024/07/13/HMI.m2024.07.13_05.00.00.jpg,1,X1.2,X
294,2024/07/13/HMI.m2024.07.13_06.00.00.jpg,1,X1.2,X
295,2024/07/13/HMI.m2024.07.13_07.00.00.jpg,1,X1.2,X
...,...,...,...,...
8211,2025/06/19/HMI.m2025.06.19_19.00.00.jpg,1,X1.9,X
8212,2025/06/19/HMI.m2025.06.19_20.00.00.jpg,1,X1.9,X
8213,2025/06/19/HMI.m2025.06.19_21.00.00.jpg,1,X1.9,X
8214,2025/06/19/HMI.m2025.06.19_22.00.00.jpg,1,X1.9,X


In [28]:
x_images["magnitude"] = pd.to_numeric(
    x_images["goes_class_exact"].str.extract(
        r"^X(\d+(?:\.\d+)?)"
    )[0],
    errors="coerce"
)

x_images

,label,goes_class_binary,goes_class_exact,category,magnitude
291,2024/07/13/HMI.m2024.07.13_03.00.00.jpg,1,X1.2,X,1.2
292,2024/07/13/HMI.m2024.07.13_04.00.00.jpg,1,X1.2,X,1.2
293,2024/07/13/HMI.m2024.07.13_05.00.00.jpg,1,X1.2,X,1.2
294,2024/07/13/HMI.m2024.07.13_06.00.00.jpg,1,X1.2,X,1.2
295,2024/07/13/HMI.m2024.07.13_07.00.00.jpg,1,X1.2,X,1.2
...,...,...,...,...,...
8211,2025/06/19/HMI.m2025.06.19_19.00.00.jpg,1,X1.9,X,1.9
8212,2025/06/19/HMI.m2025.06.19_20.00.00.jpg,1,X1.9,X,1.9
8213,2025/06/19/HMI.m2025.06.19_21.00.00.jpg,1,X1.9,X,1.9
8214,2025/06/19/HMI.m2025.06.19_22.00.00.jpg,1,X1.9,X,1.9


In [29]:
x_images["subcategory"] = (
    "X"
    + np.floor(x_images["magnitude"])
    .astype(int)
    .astype(str)
)

x_images

,label,goes_class_binary,goes_class_exact,category,magnitude,subcategory
291,2024/07/13/HMI.m2024.07.13_03.00.00.jpg,1,X1.2,X,1.2,X1
292,2024/07/13/HMI.m2024.07.13_04.00.00.jpg,1,X1.2,X,1.2,X1
293,2024/07/13/HMI.m2024.07.13_05.00.00.jpg,1,X1.2,X,1.2,X1
294,2024/07/13/HMI.m2024.07.13_06.00.00.jpg,1,X1.2,X,1.2,X1
295,2024/07/13/HMI.m2024.07.13_07.00.00.jpg,1,X1.2,X,1.2,X1
...,...,...,...,...,...,...
8211,2025/06/19/HMI.m2025.06.19_19.00.00.jpg,1,X1.9,X,1.9,X1
8212,2025/06/19/HMI.m2025.06.19_20.00.00.jpg,1,X1.9,X,1.9,X1
8213,2025/06/19/HMI.m2025.06.19_21.00.00.jpg,1,X1.9,X,1.9,X1
8214,2025/06/19/HMI.m2025.06.19_22.00.00.jpg,1,X1.9,X,1.9,X1


In [30]:
x_subcategory_counts = (
    x_images["subcategory"]
    .value_counts()
    .rename_axis("X_subcategory")
    .reset_index(name="number_of_images")
)

x_subcategory_counts["order"] = (
    x_subcategory_counts["X_subcategory"]
    .str.replace("X", "", regex=False)
    .astype(int)
)

x_subcategory_counts = (
    x_subcategory_counts
    .sort_values("order")
    .drop(columns="order")
    .reset_index(drop=True)
)

x_subcategory_counts

,X_subcategory,number_of_images
0,X1,395
1,X2,144
2,X3,24
3,X4,24
4,X7,24
5,X9,20
